In [1]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings



documents = [
    Document(
        page_content="In the 52nd T20I match between Australia and India on February 1, 2008, as part of the India tour of Australia 2007/08, India batted first and scored 74 runs, losing all 10 wickets with 3 extras. Australia chased down the target, scoring 75 runs for the loss of just 1 wicket, winning the match by 9 wickets with 52 balls remaining at the Melbourne Cricket Ground. India's playing XI included players like '7773', '7781', and '8813', while Australia's team featured '4176', '8876', and '6253'. The Man of the Match was '8876', and debutants in this game included '11984', '49327', and '48319'.",
        metadata={"source": "t20-doc"},
    ),
    Document(
        page_content="In the 54th T20I match between New Zealand and England on February 7, 2008, during the England tour of New Zealand 2007/08, England batted first and scored 193 runs, losing 8 wickets with 4 extras. New Zealand, chasing the target, scored 143 runs for the loss of 8 wickets, resulting in England winning by 50 runs at Jade Stadium in Christchurch. England's playing XI included '11556', '44660', and '8107', while New Zealand's team featured '44946', '10384', and '44930'. The Man of the Match was '2314', with debutants being '47488' and '10325'.",
        metadata={"source": "t20-doc"},
    ),
    Document(
        page_content="In the 65th T20I match, the 2nd Semi-Final between the Netherlands and Scotland on August 4, 2008, as part of the ICC World Twenty20 Qualifier, Scotland batted first and scored 107 runs, losing 8 wickets with 6 extras. The Netherlands chased down the target, scoring 110 runs for the loss of 5 wickets, winning the match by 5 wickets with 12 balls remaining at the Civil Service Cricket Club in Belfast. Scotland's playing XI included '45548', '46048', and '46142', while the Netherlands featured '10323', '48655', and '6362'. The Man of the Match was '45358', and there were no debutants in this match.",
        metadata={"source": "t20-doc"},
    ),
    Document(
        page_content="In the 66th T20I match, the 3rd Place Playoff between Kenya and Scotland on August 4, 2008, as part of the ICC World Twenty20 Qualifier, Kenya batted first and scored 106 runs, losing 9 wickets with 6 extras. Scotland chased down the target, scoring 107 runs for the loss of just 1 wicket, winning the match by 9 wickets with 11 balls remaining at the Civil Service Cricket Club in Belfast. Kenya's playing XI included '10364', '2264', and '49383', while Scotland's team featured '45548', '46048', and '46142'. The Man of the Match was '45548', and debutants included '50293'.",
        metadata={"source": "t20-doc"},
    ),
    Document(
        page_content="In the 69th T20I match between Sri Lanka and Zimbabwe on October 10, 2008, as part of the T20 Canada in Canada 2008/09, Zimbabwe batted first and scored 106 runs, losing 8 wickets with 6 extras. Sri Lanka chased down the target, scoring 107 runs for the loss of 5 wickets, winning the match by 5 wickets with 6 balls remaining at Maple Leaf North-West Ground in King City, Canada. Zimbabwe's playing XI included '10639', '10423', and '47619', while Sri Lanka's team featured '48468', '7419', and '15273'. The Man of the Match was '50377', and debutants included '50377', '47210', and '12209'.",
        metadata={"source": "t20-doc"},
    ),
]
from langchain_community.chat_models import ChatOllama

model="llama3.1"
llm = ChatOllama(model=model)


In [2]:
ollama_emb = OllamaEmbeddings(
    model=model,
)
embeddings = ollama_emb.embed_documents(
    documents
)
# r2 = ollama_emb.embed_query(
#     "What is the second letter of Greek alphabet"
# )

vectorstore = Chroma.from_documents(
    documents,
    embedding=ollama_emb,
)



5 4096


In [3]:
print(len(embeddings), len(embeddings[0]), embeddings[0])

5 4096 [1.6936668157577515, -6.407634735107422, -0.37925857305526733, 1.8356462717056274, 0.6936681866645813, -0.9074866771697998, 1.7017806768417358, 3.1287410259246826, -1.5787876844406128, 0.103966623544693, 0.4372323751449585, 1.1341549158096313, -4.231091499328613, 4.63567590713501, 2.518594980239868, 5.962555885314941, -1.325516700744629, 3.1180996894836426, -4.714583873748779, 1.924791932106018, -0.2422664314508438, 1.6955379247665405, 0.8372343182563782, 0.5741784572601318, 0.6415972709655762, 1.9340651035308838, -2.57800555229187, 3.0536837577819824, 4.010624408721924, -3.8745577335357666, -0.5569953918457031, 4.453771114349365, -0.030694622546434402, 2.1821720600128174, -0.40764251351356506, -0.6368383765220642, 0.8971760869026184, 1.4909727573394775, 0.7455653548240662, 1.8252708911895752, 2.9245879650115967, -1.8149420022964478, 2.1802074909210205, -0.18621942400932312, -0.15960244834423065, -2.5569632053375244, -1.061950922012329, 0.34541863203048706, 1.6862162351608276, 2

In [4]:
# print(vectorstore.similarity_search("cat"))

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("who won 52nd T20I match")

print(response.content)

Australia won the 52nd T20I match.
